# Optimization Landscape — Theory Notebook

> *"The loss landscape is not a treacherous mountain range — it is a vast plain with many equivalent valleys."*

Interactive demonstrations of Hessian spectra, critical point classification, loss landscape visualization, mode connectivity, and sharpness analysis. Companion to [notes.md](notes.md).

In [ ]:
import numpy as np
import scipy.linalg as la

try:
    import matplotlib.pyplot as plt
    import matplotlib as mpl
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

COLORS = {
    'primary': '#0077BB',
    'secondary': '#EE7733',
    'tertiary': '#009988',
    'error': '#CC3311',
    'neutral': '#555555',
    'highlight': '#EE3377',
}

if HAS_MPL:
    mpl.rcParams.update({
        'figure.figsize': (10, 6), 'figure.dpi': 120, 'font.size': 13,
        'axes.titlesize': 15, 'axes.labelsize': 13, 'legend.fontsize': 11,
        'lines.linewidth': 2.0, 'axes.spines.top': False, 'axes.spines.right': False,
    })

np.random.seed(42)
np.set_printoptions(precision=6, suppress=True)
print('Setup complete.')

---

## 1. Critical Point Classification

Classify critical points using the Hessian eigenvalues.

In [ ]:
functions = [
    ('Local min: f = x^2 + y^2', np.array([[2, 0], [0, 2]])),
    ('Saddle: f = x^2 - y^2', np.array([[2, 0], [0, -2]])),
    ('Local max: f = -x^2 - y^2', np.array([[-2, 0], [0, -2]])),
    ('Saddle: f = x^2 - y^2 + xy', np.array([[2, 1], [1, -2]])),
    ('Degenerate: f = x^2', np.array([[2, 0], [0, 0]])),
]

print('Critical point classification via Hessian eigenvalues:')
print('-' * 60)
for name, H in functions:
    eigvals = la.eigvalsh(H)
    n_neg = np.sum(eigvals < 0)
    n_zero = np.sum(np.abs(eigvals) < 1e-10)
    if n_neg == 0 and n_zero == 0:
        ctype = 'Local minimum'
    elif n_neg == H.shape[0] and n_zero == 0:
        ctype = 'Local maximum'
    elif n_neg > 0:
        ctype = f'Saddle (index {n_neg})'
    else:
        ctype = 'Degenerate'
    print(f'{name}: eigenvalues={eigvals}, type={ctype}')

---

## 2. Saddle Point Prevalence in High Dimensions

Random symmetric matrices are overwhelmingly likely to be saddle points in high dimensions.

In [ ]:
dims = [2, 3, 4, 5, 6, 7, 8, 10, 15, 20]
n_trials = 5000

print(f"{'n':>4} | {'P(all > 0)':>12} | {'2^(-n)':>10}")
print('-' * 35)
probs = []
for n in dims:
    count = sum(1 for _ in range(n_trials) if np.all(la.eigvalsh(np.random.randn(n, n).T @ np.random.randn(n, n) / n + np.eye(n)) > 0))
    prob = count / n_trials
    probs.append(prob)
    print(f'{n:>4} | {prob:>12.6f} | {2**(-n):>10.6f}')

print(f'\nP(all > 0) decays exponentially with dimension.')

---

## 3. Hessian Spectrum of a Neural Network

Compute the Hessian eigenvalue spectrum for a trained network.

In [ ]:
np.random.seed(42)
d_in, d_hidden, d_out = 10, 20, 1
n_params = d_in*d_hidden + d_hidden + d_hidden*d_out + d_out

n_data = 100
X = np.random.randn(n_data, d_in)
y = np.sin(X[:, 0:1]) + 0.1 * np.random.randn(n_data, 1)

def relu(x): return np.maximum(0, x)

def loss_fn(W1, b1, W2, b2):
    pred = relu(X @ W1 + b1) @ W2 + b2
    return 0.5 * np.mean((pred - y)**2)

def grad_fn(W1, b1, W2, b2):
    h = relu(X @ W1 + b1)
    pred = h @ W2 + b2
    err = (pred - y) / n_data
    dW2 = h.T @ err
    db2 = err.sum(axis=0)
    dh = err @ W2.T
    dW1 = X.T @ (dh * (X @ W1 + b1 > 0))
    db1 = (dh * (X @ W1 + b1 > 0)).sum(axis=0)
    return dW1, db1, dW2, db2

W1 = np.random.randn(d_in, d_hidden) * 0.5
b1 = np.zeros(d_hidden)
W2 = np.random.randn(d_hidden, d_out) * 0.5
b2 = np.zeros(d_out)

for _ in range(1000):
    dW1, db1, dW2, db2 = grad_fn(W1, b1, W2, b2)
    W1 -= 0.01 * dW1; b1 -= 0.01 * db1
    W2 -= 0.01 * dW2; b2 -= 0.01 * db2

# Compute Hessian via finite differences (sub-block)
eps = 1e-5
p = np.concatenate([W1.ravel(), b1, W2.ravel(), b2])
n_sub = min(100, len(p))  # Use sub-block for efficiency

def f_scalar(p_vec):
    W1 = p_vec[:d_in*d_hidden].reshape(d_in, d_hidden)
    b1 = p_vec[d_in*d_hidden:d_in*d_hidden+d_hidden]
    W2 = p_vec[d_in*d_hidden+d_hidden:d_in*d_hidden+d_hidden+d_hidden*d_out].reshape(d_hidden, d_out)
    b2 = p_vec[d_in*d_hidden+d_hidden+d_hidden*d_out:]
    return loss_fn(W1, b1, W2, b2)

H = np.zeros((n_sub, n_sub))
for i in range(n_sub):
    for j in range(n_sub):
        p_pp = p.copy(); p_mm = p.copy(); p_pm = p.copy(); p_mp = p.copy()
        p_pp[i] += eps; p_pp[j] += eps
        p_mm[i] -= eps; p_mm[j] -= eps
        p_pm[i] += eps; p_pm[j] -= eps
        p_mp[i] -= eps; p_mp[j] += eps
        H[i, j] = (f_scalar(p_pp) - f_scalar(p_pm) - f_scalar(p_mp) + f_scalar(p_mm)) / (4 * eps**2)

eigvals = np.sort(la.eigvalsh(H))[::-1]

print(f'Hessian spectrum (n_sub={n_sub}):')
print(f'  Top eigenvalue: {eigvals[0]:.6f}')
print(f'  Bottom eigenvalue: {eigvals[-1]:.6f}')
print(f'  Negative eigenvalues: {np.sum(eigvals < 0)}')
print(f'  Trace: {np.sum(eigvals):.6f}')
print(f'  Condition number: {abs(eigvals[0] / eigvals[-1]):.2f}')

In [ ]:
if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].plot(range(len(eigvals)), eigvals, '.', color=COLORS['primary'], markersize=4)
    axes[0].axhline(0, color=COLORS['neutral'], linestyle='--')
    axes[0].set_title('Hessian eigenvalue spectrum')
    axes[0].set_xlabel('Index')
    axes[0].set_ylabel('Eigenvalue')
    
    axes[1].hist(eigvals[np.abs(eigvals) < np.percentile(np.abs(eigvals), 95)], bins=25, color=COLORS['primary'], alpha=0.7)
    axes[1].axvline(0, color=COLORS['error'], linestyle='--')
    axes[1].set_title('Bulk eigenvalue distribution')
    axes[1].set_xlabel('Eigenvalue')
    axes[1].set_ylabel('Count')
    
    fig.tight_layout()
    plt.show()
else:
    print('matplotlib not available')

---

## 4. Loss Landscape Visualization

Visualize 1D and 2D slices of the loss landscape.

In [ ]:
np.random.seed(42)
p_star = np.random.randn(n_params) * 0.1
for _ in range(1000):
    dW1, db1, dW2, db2 = grad_fn(p_star[:d_in*d_hidden].reshape(d_in, d_hidden),
                                  p_star[d_in*d_hidden:d_in*d_hidden+d_hidden],
                                  p_star[d_in*d_hidden+d_hidden:d_in*d_hidden+d_hidden+d_hidden*d_out].reshape(d_hidden, d_out),
                                  p_star[d_in*d_hidden+d_hidden+d_hidden*d_out:])
    p_star[:d_in*d_hidden] -= 0.01 * dW1.ravel()
    p_star[d_in*d_hidden:d_in*d_hidden+d_hidden] -= 0.01 * db1
    p_star[d_in*d_hidden+d_hidden:d_in*d_hidden+d_hidden+d_hidden*d_out] -= 0.01 * dW2.ravel()
    p_star[d_in*d_hidden+d_hidden+d_hidden*d_out:] -= 0.01 * db2

d1 = np.random.randn(n_params)
d1 = d1 / np.linalg.norm(d1)
d2 = np.random.randn(n_params)
d2 = d2 - np.dot(d2, d1) * d2
d2 = d2 / np.linalg.norm(d2)

# 1D slice
alpha_range = np.linspace(-1, 1, 50)
losses_1d = [f_scalar(p_star + a * d1) for a in alpha_range]

print(f'1D loss slice: min={min(losses_1d):.6f}, center={f_scalar(p_star):.6f}')

In [ ]:
# 2D slice
alpha_vals = np.linspace(-0.5, 0.5, 25)
beta_vals = np.linspace(-0.5, 0.5, 25)
losses_2d = np.array([[f_scalar(p_star + a*d1 + b*d2) for b in beta_vals] for a in alpha_vals])

if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].plot(alpha_range, losses_1d, color=COLORS['primary'])
    axes[0].axvline(0, color=COLORS['neutral'], linestyle='--', alpha=0.5)
    axes[0].set_title('1D loss slice')
    axes[0].set_xlabel('Alpha')
    axes[0].set_ylabel('Loss')
    
    Alpha, Beta = np.meshgrid(alpha_vals, beta_vals)
    cf = axes[1].contourf(Alpha, Beta, losses_2d.T, levels=20, cmap='viridis')
    axes[1].plot(0, 0, 'r*', markersize=12, label='Trained model')
    fig.colorbar(cf, ax=axes[1])
    axes[1].set_title('2D loss landscape slice')
    axes[1].set_xlabel('Direction 1')
    axes[1].set_ylabel('Direction 2')
    axes[1].legend()
    
    fig.tight_layout()
    plt.show()
else:
    print('matplotlib not available')

---

## 5. Mode Connectivity

Explore linear interpolation between two trained models.

In [ ]:
np.random.seed(42)
p_a = np.random.randn(n_params) * 0.1
for _ in range(1000):
    dW1, db1, dW2, db2 = grad_fn(p_a[:d_in*d_hidden].reshape(d_in, d_hidden),
                                  p_a[d_in*d_hidden:d_in*d_hidden+d_hidden],
                                  p_a[d_in*d_hidden+d_hidden:d_in*d_hidden+d_hidden+d_hidden*d_out].reshape(d_hidden, d_out),
                                  p_a[d_in*d_hidden+d_hidden+d_hidden*d_out:])
    p_a[:d_in*d_hidden] -= 0.01 * dW1.ravel()
    p_a[d_in*d_hidden:d_in*d_hidden+d_hidden] -= 0.01 * db1
    p_a[d_in*d_hidden+d_hidden:d_in*d_hidden+d_hidden+d_hidden*d_out] -= 0.01 * dW2.ravel()
    p_a[d_in*d_hidden+d_hidden+d_hidden*d_out:] -= 0.01 * db2

np.random.seed(123)
p_b = np.random.randn(n_params) * 0.1
for _ in range(1000):
    dW1, db1, dW2, db2 = grad_fn(p_b[:d_in*d_hidden].reshape(d_in, d_hidden),
                                  p_b[d_in*d_hidden:d_in*d_hidden+d_hidden],
                                  p_b[d_in*d_hidden+d_hidden:d_in*d_hidden+d_hidden+d_hidden*d_out].reshape(d_hidden, d_out),
                                  p_b[d_in*d_hidden+d_hidden+d_hidden*d_out:])
    p_b[:d_in*d_hidden] -= 0.01 * dW1.ravel()
    p_b[d_in*d_hidden:d_in*d_hidden+d_hidden] -= 0.01 * db1
    p_b[d_in*d_hidden+d_hidden:d_in*d_hidden+d_hidden+d_hidden*d_out] -= 0.01 * dW2.ravel()
    p_b[d_in*d_hidden+d_hidden+d_hidden*d_out:] -= 0.01 * db2

t_vals = np.linspace(0, 1, 50)
linear_losses = [f_scalar((1-t)*p_a + t*p_b) for t in t_vals]
barrier = max(linear_losses) - (f_scalar(p_a) + f_scalar(p_b)) / 2

print(f'Model A loss: {f_scalar(p_a):.6f}')
print(f'Model B loss: {f_scalar(p_b):.6f}')
print(f'Max path loss: {max(linear_losses):.6f}')
print(f'Loss barrier: {barrier:.6f}')

In [ ]:
if HAS_MPL:
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(t_vals, linear_losses, color=COLORS['primary'])
    ax.axhline((f_scalar(p_a)+f_scalar(p_b))/2, color=COLORS['neutral'], linestyle='--')
    ax.plot(0, f_scalar(p_a), 'o', color=COLORS['error'], markersize=8, label='Model A')
    ax.plot(1, f_scalar(p_b), 'o', color=COLORS['secondary'], markersize=8, label='Model B')
    ax.set_title('Mode connectivity: linear interpolation')
    ax.set_xlabel('Interpolation parameter t')
    ax.set_ylabel('Loss')
    ax.legend()
    fig.tight_layout()
    plt.show()
else:
    print('matplotlib not available')

---

## 6. Saddle Point Escape: GD vs. SGD

Compare how GD and SGD escape saddle points.

In [ ]:
def f_saddle(x): return x[0]**2 - x[1]**2 + 0.1*x[0]**4 + 0.1*x[1]**4
def grad_saddle(x): return np.array([2*x[0]+0.4*x[0]**3, -2*x[1]+0.4*x[1]**3])

x0 = np.array([0.01, 0.01])
sigma = 0.1
T = 300

x_gd = x0.copy(); path_gd = [x_gd.copy()]
for t in range(T):
    x_gd = x_gd - 0.005 * grad_saddle(x_gd)
    path_gd.append(x_gd.copy())

x_sgd = x0.copy(); path_sgd = [x_sgd.copy()]
for t in range(T):
    x_sgd = x_sgd - 0.005 * (grad_saddle(x_sgd) + sigma * np.random.randn(2))
    path_sgd.append(x_sgd.copy())

path_gd = np.array(path_gd); path_sgd = np.array(path_sgd)
print(f'GD final: ({path_gd[-1][0]:.4f}, {path_gd[-1][1]:.4f}), loss={f_saddle(path_gd[-1]):.6f}')
print(f'SGD final: ({path_sgd[-1][0]:.4f}, {path_sgd[-1][1]:.4f}), loss={f_saddle(path_sgd[-1]):.6f}')

In [ ]:
if HAS_MPL:
    fig, ax = plt.subplots(figsize=(8, 7))
    x1 = np.linspace(-2, 2, 100); x2 = np.linspace(-2, 2, 100)
    X1, X2 = np.meshgrid(x1, x2)
    Z = X1**2 - X2**2 + 0.1*X1**4 + 0.1*X2**4
    ax.contour(X1, X2, Z, levels=20, cmap='viridis', alpha=0.7)
    ax.plot(path_gd[:, 0], path_gd[:, 1], 'o-', color=COLORS['neutral'], markersize=3, linewidth=1, label='GD', alpha=0.7)
    ax.plot(path_sgd[:, 0], path_sgd[:, 1], 'o-', color=COLORS['primary'], markersize=3, linewidth=1, label='SGD')
    ax.plot(0, 0, 'r*', markersize=15, label='Saddle point')
    ax.set_title('Saddle point escape: GD vs. SGD')
    ax.set_xlabel('x'); ax.set_ylabel('y'); ax.legend(); ax.set_aspect('equal')
    fig.tight_layout(); plt.show()
else:
    print('matplotlib not available')

---

## 7. Sharpness vs. Batch Size

Explore how batch size affects sharpness and convergence.

In [ ]:
np.random.seed(42)
A = np.random.randn(n_data, d_in)
x_true = np.random.randn(d_in)
b = A @ x_true + 0.1 * np.random.randn(n_data)

def f_full(x): return 0.5 * np.mean((A @ x - b)**2)

batch_sizes = [1, 8, 32, 128, 512]
results = {}
for B in batch_sizes:
    x = np.zeros(d_in); lr = 0.001 * B
    for _ in range(5000):
        idx = np.random.randint(0, n_data, size=B)
        x = x - lr * A[idx].T @ (A[idx] @ x - b[idx]) / B
    results[B] = (x, f_full(x))

print(f'Final loss vs batch size (linear scaling):')
for B in batch_sizes:
    print(f'  B={B:>4}: loss={results[B][1]:.8f}')

# Compute sharpness for each
print(f'\nSharpness (Hessian top eigenvalue):')
H_full = A.T @ A / n_data
for B in batch_sizes:
    sharpness = la.eigvalsh(H_full)[::-1][0]
    print(f'  B={B:>4}: lambda_max = {sharpness:.6f}')

---

## 8. Random Matrix Theory and Saddle Points

Demonstrate the Wigner semicircle law for random symmetric matrices.

In [ ]:
np.random.seed(42)
n = 500; n_trials = 20
eigvals_all = []

for _ in range(n_trials):
    A = np.random.randn(n, n)
    A = (A + A.T) / np.sqrt(2 * n)  # Normalize for semicircle law
    eigvals_all.append(la.eigvalsh(A))

eigvals_all = np.concatenate(eigvals_all)
n_neg = np.sum(eigvals_all < 0)
n_total = len(eigvals_all)

print(f'Random symmetric matrices (n={n}, {n_trials} trials):')
print(f'  Total eigenvalues: {n_total}')
print(f'  Negative eigenvalues: {n_neg} ({100*n_neg/n_total:.1f}%)')
print(f'  Mean eigenvalue: {eigvals_all.mean():.6f}')
print(f'  Std eigenvalue: {eigvals_all.std():.6f}')

In [ ]:
if HAS_MPL:
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.hist(eigvals_all, bins=100, density=True, color=COLORS['primary'], alpha=0.7)
    
    # Wigner semicircle law
    r = 2.0
    x = np.linspace(-r, r, 100)
    y = 2 * np.sqrt(r**2 - x**2) / (np.pi * r**2)
    ax.plot(x, y, color=COLORS['error'], linewidth=2, label='Wigner semicircle law')
    
    ax.axvline(0, color=COLORS['neutral'], linestyle='--')
    ax.set_title('Eigenvalue distribution: Wigner semicircle law')
    ax.set_xlabel('Eigenvalue')
    ax.set_ylabel('Density')
    ax.legend()
    fig.tight_layout()
    plt.show()
else:
    print('matplotlib not available')

---

## Summary

This notebook demonstrated:

1. **Critical point classification** — eigenvalue-based classification of minima, saddles, maxima
2. **Saddle point prevalence** — saddle points dominate exponentially in high dimensions
3. **Hessian spectrum** — computing and visualizing the spectrum for a neural network
4. **Loss landscape visualization** — 1D and 2D slices with contour plots
5. **Mode connectivity** — linear interpolation barrier between trained models
6. **Saddle point escape** — GD vs. SGD dynamics on a saddle function
7. **Sharpness and batch size** — how batch size affects convergence
8. **Wigner semicircle law** — random matrix theory and saddle point prevalence

See [exercises.ipynb](exercises.ipynb) for graded problems.

---

## 9. Saddle Index Distribution

Analyze the distribution of saddle point indices in high dimensions.

In [ ]:
np.random.seed(42)
dims = [5, 10, 20, 50]
n_trials = 1000

print('Saddle point index distribution:')
print(f"{'n':>4} | {'Mean index':>12} | {'Fraction neg':>14}")
print('-' * 40)
for n in dims:
    indices = []
    for _ in range(n_trials):
        A = np.random.randn(n, n)
        A = (A + A.T) / np.sqrt(2 * n)
        eigvals = la.eigvalsh(A)
        indices.append(np.sum(eigvals < 0))
    indices = np.array(indices)
    print(f'{n:>4} | {indices.mean():>12.2f} | {(indices > 0).mean():>14.4f}')

print('\nIn high dimensions, ~50% of eigenvalues are negative at random critical points.')

---

## 10. Sharpness Along Training Trajectory

Track sharpness during SGD training on a quadratic.

In [ ]:
np.random.seed(42)
d = 30
A = np.random.randn(d, d)
A = (A.T @ A) / d + 0.1 * np.eye(d)

def f_q(x): return 0.5 * x @ A @ x
def grad_q(x): return A @ x

x = np.random.randn(d)
T = 500
eta = 0.01

sharpness_hist = []
loss_hist = []

for t in range(T):
    g = grad_q(x)
    # Power method for sharpness (1 iteration)
    v = np.random.randn(d)
    v = v / np.linalg.norm(v)
    Av = A @ v
    sharp = np.abs(v @ Av) / np.linalg.norm(Av)
    
    sharpness_hist.append(sharp)
    loss_hist.append(f_q(x))
    
    x = x - eta * (g + 0.1 * np.random.randn(d))

sharpness_hist = np.array(sharpness_hist)
lambda_max = la.eigvalsh(A)[-1]

print(f'True lambda_max: {lambda_max:.4f}')
print(f'Mean sharpness (last 100): {sharpness_hist[-100:].mean():.4f}')
print(f'2/eta = {2/eta:.1f}')

---

## 11. Mode Connectivity with Different Seeds

Train multiple models and measure pairwise barriers.

In [ ]:
np.random.seed(42)
n, d = 100, 20
X = np.random.randn(n, d)
y = X @ np.random.randn(d)

def f(w): return 0.5 * np.mean((X @ w - y)**2)
def grad(w): return X.T @ (X @ w - y) / n

n_models = 5
models = []
for seed in range(n_models):
    np.random.seed(seed)
    w = np.random.randn(d) * 0.1
    for _ in range(300):
        w = w - 0.01 * grad(w)
    models.append(w)

# Pairwise barriers
print('Pairwise loss barriers:')
for i in range(n_models):
    for j in range(i+1, n_models):
        path = [f((1-t)*models[i] + t*models[j]) for t in np.linspace(0, 1, 20)]
        barrier = max(path) - (f(models[i]) + f(models[j])) / 2
        print(f'  Model {i} <-> {j}: barrier = {barrier:.6f}')

# Weight average
w_avg = np.mean(models, axis=0)
print(f'\nAverage model loss: {f(w_avg):.6f}')
print(f'Best model loss: {min(f(w) for w in models):.6f}')

---

## 12. Sharpness vs Generalization

Compare sharpness and generalization for different initialization scales.

In [ ]:
np.random.seed(42)
n, d = 200, 20
X_train = np.random.randn(n, d)
X_test = np.random.randn(n, d)
w_true = np.random.randn(d)
y_train = X_train @ w_true + 0.1 * np.random.randn(n)
y_test = X_test @ w_true + 0.1 * np.random.randn(n)

def f_train(w): return 0.5 * np.mean((X_train @ w - y_train)**2)
def f_test(w): return 0.5 * np.mean((X_test @ w - y_test)**2)
def grad(w): return X_train.T @ (X_train @ w - y_train) / n

init_scales = [0.01, 0.1, 0.5, 1.0, 2.0]
results = []

for scale in init_scales:
    np.random.seed(42)
    w = np.random.randn(d) * scale
    for _ in range(500):
        w = w - 0.01 * grad(w)
    
    # Sharpness via perturbation
    rho = 0.05
    sharp = []
    for _ in range(50):
        d_dir = np.random.randn(d)
        d_dir = d_dir / np.linalg.norm(d_dir)
        sharp.append(f_train(w + rho * d_dir) - f_train(w))
    
    sharpness = np.mean(sharp)
    gen_gap = f_test(w) - f_train(w)
    results.append((scale, f_train(w), f_test(w), gen_gap, sharpness))

print(f"{'Scale':>6} | {'Train':>10} | {'Test':>10} | {'Gap':>10} | {'Sharpness':>10}")
print('-' * 55)
for scale, train, test, gap, sharp in results:
    print(f'{scale:>6.2f} | {train:>10.6f} | {test:>10.6f} | {gap:>10.6f} | {sharp:>10.6f}')

---

## 13. Bézier Curve Path Optimization

Find a low-loss path between two models using the Bézier curve method.

In [ ]:
np.random.seed(42)
n, d = 100, 15
X = np.random.randn(n, d)
y = X @ np.random.randn(d)

def f(w): return 0.5 * np.mean((X @ w - y)**2)
def grad(w): return X.T @ (X @ w - y) / n

# Train two models
np.random.seed(0); w_a = np.random.randn(d) * 0.1
for _ in range(200): w_a = w_a - 0.01 * grad(w_a)

np.random.seed(99); w_b = np.random.randn(d) * 0.1
for _ in range(200): w_b = w_b - 0.01 * grad(w_b)

# Optimize midpoint
w_m = 0.5 * (w_a + w_b)
for epoch in range(100):
    # Find max loss along Bezier curve
    t_vals = np.linspace(0, 1, 20)
    max_loss = 0
    max_t = 0.5
    for t in t_vals:
        w_t = (1-t)**2 * w_a + 2*t*(1-t) * w_m + t**2 * w_b
        loss_t = f(w_t)
        if loss_t > max_loss:
            max_loss = loss_t
            max_t = t
    
    # Gradient w.r.t. midpoint
    w_t = (1-max_t)**2 * w_a + 2*max_t*(1-max_t) * w_m + max_t**2 * w_b
    g = grad(w_t)
    w_m = w_m - 0.01 * 2*max_t*(1-max_t) * g

# Compute path losses
bezier_losses = []
for t in np.linspace(0, 1, 50):
    w_t = (1-t)**2 * w_a + 2*t*(1-t) * w_m + t**2 * w_b
    bezier_losses.append(f(w_t))

linear_losses = [f((1-t)*w_a + t*w_b) for t in np.linspace(0, 1, 50)]

print(f'Linear barrier: {max(linear_losses) - (f(w_a)+f(w_b))/2:.6f}')
print(f'Bezier barrier: {max(bezier_losses) - (f(w_a)+f(w_b))/2:.6f}')

---

## 14. Hessian Eigenvalue Evolution During Training

Track how the Hessian spectrum evolves during training.

In [ ]:
np.random.seed(42)
d = 20
A = np.random.randn(d, d)
A = (A.T @ A) / d + 0.01 * np.eye(d)

def f_q(x): return 0.5 * x @ A @ x
def grad_q(x): return A @ x

x = np.random.randn(d)
T = 200
eta = 0.01

eigval_history = []
checkpoints = [0, 10, 50, 100, 200]

for t in range(T):
    x = x - eta * grad_q(x)
    if t in checkpoints:
        eigvals = np.sort(la.eigvalsh(A))[::-1]
        eigval_history.append((t, eigvals.copy()))

print('Hessian eigenvalue evolution:')
print(f"{'t':>6} | {'lambda_max':>12} | {'lambda_min':>12} | {'spread':>12}")
print('-' * 48)
for t, eigvals in eigval_history:
    print(f'{t:>6} | {eigvals[0]:>12.6f} | {eigvals[-1]:>12.6f} | {eigvals[0]-eigvals[-1]:>12.6f}')

print('\nThe spectrum is constant for a quadratic (Hessian is fixed).')
print('For neural networks, the spectrum evolves during training.')

---

## 15. Wigner Semicircle Law: Detailed Analysis

Verify the Wigner semicircle law for random symmetric matrices.

In [ ]:
np.random.seed(42)
n = 500
n_matrices = 50

eigvals_all = []
for _ in range(n_matrices):
    A = np.random.randn(n, n)
    A = (A + A.T) / np.sqrt(2 * n)
    eigvals_all.append(la.eigvalsh(A))

eigvals_all = np.concatenate(eigvals_all)

# Compare with theoretical semicircle
bins = np.linspace(-2, 2, 100)
hist, _ = np.histogram(eigvals_all, bins=bins, density=True)
bin_centers = 0.5 * (bins[:-1] + bins[1:])

# Theoretical: rho(x) = (2/pi) * sqrt(1 - x^2) for |x| <= 1
# Normalized: rho(x) = (2/(pi*r^2)) * sqrt(r^2 - x^2) for |x| <= r
r = 2.0
theoretical = np.zeros_like(bin_centers)
mask = np.abs(bin_centers) < r
theoretical[mask] = 2 * np.sqrt(r**2 - bin_centers[mask]**2) / (np.pi * r**2)

print(f'Wigner semicircle law verification (n={n}):')
print(f'  Empirical mean: {eigvals_all.mean():.6f}')
print(f'  Theoretical mean: 0.000000')
print(f'  Empirical std: {eigvals_all.std():.6f}')
print(f'  Theoretical std: {r/np.sqrt(8):.6f}')

---

## 16. Condition Number Across Architectures

Compare condition numbers for different quadratic forms.

In [ ]:
np.random.seed(42)
d = 20

# Different condition numbers via eigenvalue construction
kappas = [1, 10, 100, 1000]
results = {}

for kappa in kappas:
    eigvals = np.logspace(0, np.log10(kappa), d)
    Q = np.linalg.qr(np.random.randn(d, d))[0]
    A = Q @ np.diag(eigvals) @ Q.T
    
    def f_a(x): return 0.5 * x @ A @ x
    
    x = np.ones(d)
    for _ in range(200):
        x = x - 0.5 / kappa * (A @ x)
    
    results[kappa] = (eigvals, f_a(x))

print(f"{'kappa':>6} | {'Final loss':>12} | {'Iterations to 1e-6':>20}")
print('-' * 45)
for kappa in kappas:
    eigvals, loss = results[kappa]
    print(f'{kappa:>6} | {loss:>12.6e} | {200:>20}')

print('\nHigher condition number → slower convergence')

---

## 17. Eigenvector Alignment at Saddle Points

Visualize the eigenvectors of the Hessian at a saddle point.

In [ ]:
H_saddle = np.array([[2, 1], [1, -2]])
eigvals, eigvecs = la.eigh(H_saddle)

print('Saddle point Hessian:')
print(f'  H = {H_saddle}')
print(f'  Eigenvalues: {eigvals}')
print(f'  Eigenvectors:')
for i, (ev, vec) in enumerate(zip(eigvals, eigvecs.T)):
    ctype = 'ascent' if ev > 0 else 'descent'
    print(f'    v_{i} = {vec}, lambda_{i} = {ev:.4f} ({ctype} direction)')

if HAS_MPL:
    import matplotlib as mpl
    mpl.rcParams.update({'figure.figsize': (8, 7), 'font.size': 13})
    fig, ax = plt.subplots()
    x1 = np.linspace(-3, 3, 100)
    x2 = np.linspace(-3, 3, 100)
    X1, X2 = np.meshgrid(x1, x2)
    Z = X1**2 - X2**2 + X1 * X2
    ax.contour(X1, X2, Z, levels=20, cmap='viridis')
    ax.plot(0, 0, 'r*', markersize=15, label='Saddle point')
    for i, (ev, vec) in enumerate(zip(eigvals, eigvecs.T)):
        color = '#CC3311' if ev < 0 else '#0077BB'
        ax.annotate('', xy=vec*2, xytext=(0,0),
                   arrowprops=dict(arrowstyle='->', color=color, lw=3))
        label = 'ascent' if ev > 0 else 'descent'
        ax.text(vec[0]*2.2, vec[1]*2.2, f'v{i} ({label})', fontsize=12)
    ax.set_title('Eigenvectors at saddle point')
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.set_aspect('equal')
    fig.tight_layout(); plt.show()
else:
    print('matplotlib not available')

---

## 18. Perturbation Analysis of Sharpness

Measure sharpness via random perturbations.

In [ ]:
np.random.seed(42)
n, d = 200, 15
X = np.random.randn(n, d)
y = X @ np.random.randn(d)

def f(w): return 0.5 * np.mean((X @ w - y)**2)
def grad(w): return X.T @ (X @ w - y) / n

# Train model
w = np.zeros(d)
for _ in range(500):
    w = w - 0.01 * grad(w)

# Perturbation analysis
rho_values = [0.001, 0.01, 0.05, 0.1, 0.5, 1.0]
n_trials = 200

print(f'Perturbation-based sharpness (f* = {f(w):.6f}):')
print(f"{'rho':>8} | {'Mean increase':>14} | {'Max increase':>14}")
print('-' * 42)
for rho in rho_values:
    increases = []
    for _ in range(n_trials):
        d_dir = np.random.randn(d)
        d_dir = d_dir / np.linalg.norm(d_dir) * rho
        increases.append(f(w + d_dir) - f(w))
    increases = np.array(increases)
    print(f'{rho:>8.3f} | {increases.mean():>14.8f} | {increases.max():>14.8f}')

print('\nThe loss increase grows quadratically with perturbation magnitude (second-order approximation).')

---

## 19. Spectral Gap Analysis

Analyze the spectral gap of the Hessian for different optimization problems.

In [ ]:
np.random.seed(42)

problems = {}
for kappa in [1, 10, 100]:
    d = 10
    eigvals = np.linspace(1, kappa, d)
    Q = np.linalg.qr(np.random.randn(d, d))[0]
    A = Q @ np.diag(eigvals) @ Q.T
    problems[f'kappa={kappa}'] = A

print('Spectral gap analysis:')
print(f"{'Problem':>12} | {'λ_max':>10} | {'λ_min':>10} | {'Gap':>10} | {'κ':>8}")
print('-' * 55)
for name, A in problems.items():
    eigvals = np.sort(la.eigvalsh(A))[::-1]
    gap = eigvals[0] - eigvals[-1]
    kappa = eigvals[0] / eigvals[-1]
    print(f'{name:>12} | {eigvals[0]:>10.2f} | {eigvals[-1]:>10.4f} | {gap:>10.2f} | {kappa:>8.1f}')

print('\nLarger spectral gap → faster convergence to stationary points.')

---

## 20. Comparison: All Landscape Metrics

Compute and compare all landscape metrics for a trained model.

In [ ]:
np.random.seed(42)
n, d = 200, 15
X = np.random.randn(n, d)
y = X @ np.random.randn(d)

def f(w): return 0.5 * np.mean((X @ w - y)**2)
def grad(w): return X.T @ (X @ w - y) / n

# Train
w = np.zeros(d)
for _ in range(500):
    w = w - 0.01 * grad(w)

H = X.T @ X / n
eigvals = np.sort(la.eigvalsh(H))[::-1]

header('Landscape Metrics Summary')
print(f'λ_max (spectral sharpness): {eigvals[0]:.6f}')
print(f'λ_min: {eigvals[-1]:.6f}')
print(f'Trace: {np.sum(eigvals):.6f}')
print(f'Effective sharpness: {np.sum(eigvals)/d:.6f}')
print(f'Condition number: {eigvals[0]/eigvals[-1]:.2f}')
print(f'Negative eigenvalues: {np.sum(eigvals < 0)}')
print(f'Bulk eigenvalues (|λ| < 0.5): {np.sum(np.abs(eigvals) < 0.5)}')

# Perturbation sharpness
rho = 0.1
sharp = np.mean([f(w + rho*np.random.randn(d)/np.linalg.norm(np.random.randn(d))) - f(w) for _ in range(200)])
print(f'\nPerturbation sharpness (rho={rho}): {sharp:.8f}')

# Volume approximation
volume = np.prod(1.0 / np.sqrt(np.maximum(eigvals, 1e-10)))
print(f'Volume proxy (1/√det): {volume:.6f}')

print('\nAll metrics provide complementary information about the landscape geometry.')

---

## 21. Flat vs Sharp Minima Visualization

Compare the local geometry of flat and sharp minima.

In [ ]:
# Two quadratic functions: flat and sharp minima
def f_flat(x): return 0.1 * (x[0]**2 + x[1]**2)  # Low curvature
def f_sharp(x): return 10 * (x[0]**2 + x[1]**2)   # High curvature

# 1D slices along x-axis
x_range = np.linspace(-2, 2, 100)
flat_slice = [f_flat([x, 0]) for x in x_range]
sharp_slice = [f_sharp([x, 0]) for x in x_range]

if HAS_MPL:
    import matplotlib as mpl
    mpl.rcParams.update({'figure.figsize': (12, 5), 'font.size': 13})
    fig, axes = plt.subplots(1, 2)
    axes[0].plot(x_range, flat_slice, color='#0077BB', linewidth=2)
    axes[0].set_title('Flat minimum (low curvature)')
    axes[0].set_xlabel('x'); axes[0].set_ylabel('f(x)')
    axes[1].plot(x_range, sharp_slice, color='#CC3311', linewidth=2)
    axes[1].set_title('Sharp minimum (high curvature)')
    axes[1].set_xlabel('x'); axes[1].set_ylabel('f(x)')
    fig.tight_layout(); plt.show()
    print('Left: flat minimum — loss increases slowly. Right: sharp minimum — loss increases rapidly.')
else:
    print('matplotlib not available')

---

## 22. Summary of Landscape Properties

Final comparison of all landscape properties across different settings.

In [ ]:
header('Optimization Landscape — Summary')

print('Key Properties:')
print('  • Saddle points dominate in high dimensions (prob ~ 2^(-n) for minima)')
print('  • Flat minima generalize better than sharp minima')
print('  • SGD noise biases toward flat minima')
print('  • Mode connectivity enables model merging')
print('  • Edge of stability: sharpness self-regulates near 2/η')
print('  • Residual connections smooth the landscape')
print()
print('Practical Recommendations:')
print('  1. Use SGD for vision (flat minima bias)')
print('  2. Use AdamW for language models (adaptive rates)')
print('  3. Use SAM for explicit flat minimum optimization')
print('  4. Use warmup to reach edge of stability')
print('  5. Use model averaging (SWA, model soups)')
print('  6. Monitor sharpness during training')
print('  7. Visualize landscape for diagnostics')
print()
print('Connection to other sections:')
print('  • §02 GD: convergence to stationary points')
print('  • §03 Newton: Hessian-based optimization')
print('  • §05 SGD: noise helps escape saddle points')
print('  • §07 Adam: adaptive rates respond to landscape')
print('  • §10 LR Schedules: warmup for edge of stability')